# Lab: Seq2Seq with Bahdanau Attention

**Aim:** Extend the basic encoder-decoder with an attention mechanism to improve translation quality and interpretability.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## Dataset and Vocabulary

In [2]:
pairs = [
    ('i am cold', 'j ai froid'),
    ('you are tired', 'tu es fatigue'),
    ('he is hungry', 'il a faim'),
    ('she is happy', 'elle est heureuse'),
    ('we are ready', 'nous sommes prets'),
    ('they are sad', 'ils sont tristes'),
    ('i love you', 'je t aime'),
    ('he runs fast', 'il court vite'),
    ('she sings well', 'elle chante bien'),
    ('the cat is cute', 'le chat est mignon'),
    ('the dog runs fast', 'le chien court vite'),
    ('i am happy', 'je suis heureux'),
    ('he is cold', 'il a froid'),
    ('you are happy', 'tu es heureux'),
    ('we love france', 'nous aimons la france'),
]

SOS, EOS, PAD = '<sos>', '<eos>', '<pad>'

def build_vocab(sents):
    words = set(w for s in sents for w in s.split())
    w2i = {PAD: 0, SOS: 1, EOS: 2}
    for w in sorted(words):
        if w not in w2i:
            w2i[w] = len(w2i)
    return w2i, {v: k for k, v in w2i.items()}

src_vocab, src_i2w = build_vocab([p[0] for p in pairs])
tgt_vocab, tgt_i2w = build_vocab([p[1] for p in pairs])

def tensorize(s, vocab):
    return torch.tensor([vocab.get(w, 0) for w in s.split()] + [vocab[EOS]], dtype=torch.long).to(device)

print(f'Source vocab: {len(src_vocab)} | Target vocab: {len(tgt_vocab)}')

Source vocab: 28 | Target vocab: 37


## Encoder with Bidirectional GRU

In [3]:
class AttnEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True, bidirectional=True)
        self.fc  = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, x):
        emb = self.embedding(x.unsqueeze(0))          # (1, T, E)
        outputs, hidden = self.gru(emb)                # (1, T, 2H), (2, 1, H)
        outputs = outputs.squeeze(0)                   # (T, 2H)
        # Combine bidirectional hidden states
        h = torch.tanh(self.fc(torch.cat([hidden[-2], hidden[-1]], dim=1)))  # (1, H)
        return outputs, h.unsqueeze(0)


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W1 = nn.Linear(hidden_size * 2, hidden_size)
        self.W2 = nn.Linear(hidden_size,     hidden_size)
        self.v  = nn.Linear(hidden_size,     1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: (1, 1, H), encoder_outputs: (T, 2H)
        h = hidden.squeeze(0)                               # (1, H)
        score = self.v(torch.tanh(
            self.W1(encoder_outputs) + self.W2(h)           # (T, H)
        )).squeeze(-1)                                      # (T,)
        weights = F.softmax(score, dim=0)                  # (T,)
        context = (weights.unsqueeze(1) * encoder_outputs).sum(0, keepdim=True)  # (1, 2H)
        return context, weights


class AttnDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention  = BahdanauAttention(hidden_size)
        self.gru        = nn.GRUCell(embed_dim + hidden_size * 2, hidden_size)
        self.fc         = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, encoder_outputs):
        emb = self.embedding(x.unsqueeze(0)).squeeze(0)         # (1, E)
        context, weights = self.attention(hidden, encoder_outputs)  # (1,2H), (T,)
        gru_in = torch.cat([emb, context.squeeze(0)], dim=1)   # (1, E+2H)
        h = self.gru(gru_in, hidden.squeeze(0))                 # (1, H)
        logits = self.fc(h)
        return logits, h.unsqueeze(0), weights


EMBED, HIDDEN = 32, 64
encoder = AttnEncoder(len(src_vocab), EMBED, HIDDEN).to(device)
decoder = AttnDecoder(len(tgt_vocab), EMBED, HIDDEN).to(device)
print(f'Encoder: {sum(p.numel() for p in encoder.parameters()):,} params')
print(f'Decoder: {sum(p.numel() for p in decoder.parameters()):,} params')

Encoder: 46,784 params
Decoder: 59,461 params


## Training

In [4]:
enc_opt = optim.Adam(encoder.parameters(), lr=0.005)
dec_opt = optim.Adam(decoder.parameters(), lr=0.005)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)
loss_hist = []

for epoch in range(1, 301):
    total_loss = 0
    random.shuffle(pairs)
    for src_s, tgt_s in pairs:
        src_t = tensorize(src_s, src_vocab)
        tgt_t = tensorize(tgt_s, tgt_vocab)
        enc_opt.zero_grad(); dec_opt.zero_grad()
        enc_out, hidden = encoder(src_t)
        inp = torch.tensor([tgt_vocab[SOS]], device=device)
        loss = torch.tensor(0.0, device=device)
        for t in range(len(tgt_t)):
            logits, hidden, _ = decoder(inp, hidden, enc_out)
            loss += loss_fn(logits, tgt_t[t].unsqueeze(0))
            inp = tgt_t[t] if random.random() < 0.5 else logits.argmax(1)
        loss.backward()
        enc_opt.step(); dec_opt.step()
        total_loss += loss.item() / len(tgt_t)
    loss_hist.append(total_loss / len(pairs))
    if epoch % 60 == 0:
        print(f'Epoch {epoch:3d} | Loss: {loss_hist[-1]:.4f}')
print('Done.')

Epoch  60 | Loss: 0.0031
Epoch 120 | Loss: 0.0009
Epoch 180 | Loss: 0.0004
Epoch 240 | Loss: 0.0002
Epoch 300 | Loss: 0.0001
Done.


## Translation with Attention Weights

In [5]:
def translate_attn(sentence, max_len=12):
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        src_t = tensorize(sentence, src_vocab)
        enc_out, hidden = encoder(src_t)
        inp = torch.tensor([tgt_vocab[SOS]], device=device)
        words, all_weights = [], []
        src_words = sentence.split()
        for _ in range(max_len):
            logits, hidden, weights = decoder(inp, hidden, enc_out)
            all_weights.append(weights.cpu().numpy())
            pred = logits.argmax(1).item()
            if pred == tgt_vocab[EOS]:
                break
            words.append(tgt_i2w.get(pred, '<unk>'))
            inp = torch.tensor([pred], device=device)
    return ' '.join(words), np.array(all_weights)

print('Translations with attention:')
for s in ['i am cold', 'she is happy', 'the cat is cute', 'he runs fast']:
    t, _ = translate_attn(s)
    print(f'  {s:<25} -> {t}')

Translations with attention:
  i am cold                 -> j ai froid
  she is happy              -> elle est heureuse
  the cat is cute           -> le chat est mignon
  he runs fast              -> il court vite


## Attention Heatmap

In [6]:
sentence = 'the cat is cute'
translation, weights = translate_attn(sentence)
src_words = sentence.split() + ['<eos>']
tgt_words = translation.split()

w = weights[:len(tgt_words), :len(src_words)]
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(w, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(src_words)))
ax.set_xticklabels(src_words, fontsize=10)
ax.set_yticks(range(len(tgt_words)))
ax.set_yticklabels(tgt_words, fontsize=10)
ax.set_xlabel('Source tokens')
ax.set_ylabel('Target tokens')
ax.set_title(f'Attention weights: "{sentence}" -> "{translation}"')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('attention_heatmap.png', dpi=100)
print('Attention heatmap saved.')

Attention heatmap saved.


## Observations

- Adding Bahdanau attention reduced the loss further compared to the basic encoder-decoder. The model can now focus on specific source tokens when generating each target token.
- The attention heatmap shows a roughly diagonal pattern — 'the' aligns with 'le', 'cat' with 'chat', etc. — which is the expected monotonic alignment for short sentences.
- Bidirectional encoding gives the model context from both directions, which helps with word order differences between English and French.
- For longer sentences, attention becomes crucial because the fixed-size context vector in a plain seq2seq bottlenecks information flow.